# Generate Realistic UPI Transaction Dataset - 500K Records

**Enhanced baseline solution data generation using original Kaggle script**

## Features Generated (17 total):
1. **Core Metadata**: Transaction ID, Timestamp, Amount
2. **Payment Context**: Type (P2P/P2M/Bill/Recharge), Merchant Category (10 types)
3. **Demographics**: Sender/Receiver Age Groups (5 categories)
4. **Geographic**: Sender State (10 major states)
5. **Banking**: Sender/Receiver Banks (8 major banks)
6. **Technology**: Device Type (Android/iOS/Web), Network Type (4G/5G/WiFi/3G)
7. **Risk**: Fraud Flag (0.2% fraud rate)
8. **Derived**: Hour of Day, Day of Week, Is Weekend

## Realistic Patterns:
- **Temporal**: Peak hours 10AM-1PM & 5-9PM (7PM peak at 8.36%)
- **Age-Based Spending**: 18-25 spend 1.3x on entertainment, 56+ spend 2.2x on healthcare
- **Geographic**: Maharashtra 15%, UP 12% (matches population)
- **Banking**: SBI 25% market share (matches RBI data)

**Output**: 500,000 transactions → workspace.bronze.fraud_data

In [0]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 SURAKSHA - REALISTIC DATA GENERATION")
print("="*70)
print("\n✓ Libraries imported successfully")

In [0]:
def hourly_distribution():
    '''Return realistic hourly distribution for UPI transactions
    Peak hours: 10-13 and 17-21
    Hour 19 (7:00 PM) shows highest probability at 0.0836
    Hours 03-04 (3:00-4:00 AM) have lowest at 0.0047
    '''
    hours_prob = [
        0.013942,  # 00:00
        0.009293,  # 01:00
        0.007435,  # 02:00
        0.004647,  # 03:00 (lowest)
        0.004647,  # 04:00 (lowest)
        0.007435,  # 05:00
        0.013941,  # 06:00
        0.023234,  # 07:00
        0.032528,  # 08:00
        0.041822,  # 09:00
        0.055762,  # 10:00 (morning peak)
        0.065056,  # 11:00
        0.069703,  # 12:00
        0.060409,  # 13:00
        0.046468,  # 14:00
        0.051115,  # 15:00
        0.055762,  # 16:00
        0.074349,  # 17:00 (evening peak start)
        0.078996,  # 18:00
        0.083643,  # 19:00 (PEAK - 7:00 PM)
        0.074349,  # 20:00
        0.065056,  # 21:00
        0.037175,  # 22:00
        0.023234,  # 23:00
    ]
    
    # Ensure probabilities sum to exactly 1.0
    rounded = np.round(hours_prob, 3)
    rounded[np.argmax(rounded)] += 1.0 - rounded.sum()
    
    return rounded

print("\n📊 Hourly distribution function defined")
print(f"   Total probability: {np.sum(hourly_distribution())}")
print(f"   Peak hour: 19:00 (7 PM) - {hourly_distribution()[19]:.4f}")
print(f"   Low hour: 03:00 (3 AM) - {hourly_distribution()[3]:.4f}")

In [0]:
def get_amt(merchant_categories, age_groups):
    '''Generate realistic transaction amounts based on merchant category and age group
    
    Age-based spending multipliers reflect real consumer behavior:
    - 18-25: High entertainment (1.3x), low healthcare (0.6x)
    - 26-35: Education focus (1.2x), balanced spending
    - 36-45: Peak family expenses, education (1.5x), grocery (1.2x)
    - 46-55: Healthcare rises (1.4x), entertainment drops (0.8x)
    - 56+: Healthcare dominant (2.2x), minimal entertainment (0.65x)
    '''
    amounts = []
    
    # Age-based spending multipliers for each category
    age_multipliers = {
        '18-25': {
            'Food': 1.15,           # Young people dine out more frequently
            'Grocery': 0.85,        # Smaller household needs
            'Fuel': 0.90,          # Less vehicle ownership
            'Entertainment': 1.30,  # High entertainment spending
            'Shopping': 1.25,       # Fashion and lifestyle focus
            'Healthcare': 0.60,     # Minimal health issues
            'Education': 0.70,      # Personal learning, not family education
            'Transport': 1.20,      # High mobility, ride-sharing
            'Utilities': 0.70,      # Shared living, lower bills
            'Other': 1.10          # Miscellaneous young adult spending
        },
        '26-35': {
            'Food': 1.10,           # Continued dining out but moderating
            'Grocery': 1.05,        # Establishing households
            'Fuel': 1.00,          # Standard vehicle usage
            'Entertainment': 1.15,  # Still active entertainment spending
            'Shopping': 1.15,       # Fashion and lifestyle purchases
            'Healthcare': 0.85,     # Increasing but still moderate
            'Education': 1.20,      # Professional development, early parenting
            'Transport': 1.10,      # Commuting and lifestyle
            'Utilities': 0.95,      # Moderate household expenses
            'Other': 1.05          # General adult spending
        },
        '36-45': {
            'Food': 1.00,           # Balanced family dining
            'Grocery': 1.20,        # Peak family shopping needs
            'Fuel': 1.15,          # Family vehicle usage
            'Entertainment': 0.95,  # Family-oriented entertainment
            'Shopping': 1.00,       # Household and family needs
            'Healthcare': 1.15,     # Increasing health consciousness
            'Education': 1.50,      # Peak children's education spending
            'Transport': 1.05,      # Family transport needs
            'Utilities': 1.20,      # Full household utility expenses
            'Other': 1.00          # Standard family spending
        },
        '46-55': {
            'Food': 0.95,           # Reduced dining out
            'Grocery': 1.15,        # Continued family needs
            'Fuel': 1.10,          # Maintained vehicle usage
            'Entertainment': 0.80,  # Reduced entertainment spending
            'Shopping': 0.85,       # Reduced fashion spending
            'Healthcare': 1.40,     # Significant increase in health spending
            'Education': 1.30,      # Continued children's education
            'Transport': 0.95,      # Reduced leisure travel
            'Utilities': 1.35,      # Higher utility consumption
            'Other': 0.90          # More conservative spending
        },
        '56+': {
            'Food': 0.85,           # Limited dining out
            'Grocery': 1.00,        # Stable grocery needs
            'Fuel': 0.80,          # Reduced mobility
            'Entertainment': 0.65,  # Minimal entertainment spending
            'Shopping': 0.70,       # Essential purchases only
            'Healthcare': 2.20,     # Highest healthcare spending
            'Education': 0.80,      # Grandchildren or personal interest
            'Transport': 0.75,      # Limited transport needs
            'Utilities': 1.50,      # Higher utility usage due to home presence
            'Other': 0.80          # Conservative miscellaneous spending
        }
    }
    
    for i, category in enumerate(merchant_categories):
        age_group = age_groups[i] if isinstance(age_groups, (list, np.ndarray)) else age_groups
        
        if category == 'Food':
            # Food/Restaurant: 50-2000, concentrated around 200-800
            rand = np.random.random()
            if rand < 0.40:
                amount = np.random.lognormal(mean=np.log(120), sigma=0.6)  # 50-200
            elif rand < 0.80:
                amount = np.random.lognormal(mean=np.log(400), sigma=0.5)  # 200-800
            else:
                amount = np.random.lognormal(mean=np.log(1200), sigma=0.4)  # 800-2000
                
        elif category == 'Grocery':
            # Grocery: 100-3000, concentrated around 400-1500
            rand = np.random.random()
            if rand < 0.30:
                amount = np.random.lognormal(mean=np.log(250), sigma=0.5)  # 100-400
            elif rand < 0.70:
                amount = np.random.lognormal(mean=np.log(800), sigma=0.6)  # 400-1500
            else:
                amount = np.random.lognormal(mean=np.log(2000), sigma=0.4)  # 1500-3000
                
        elif category == 'Fuel':
            # Fuel: 300-5000, based on vehicle type
            rand = np.random.random()
            if rand < 0.40:
                amount = np.random.lognormal(mean=np.log(500), sigma=0.3)  # 300-800 (two-wheeler)
            elif rand < 0.70:
                amount = np.random.lognormal(mean=np.log(1200), sigma=0.4)  # 800-2000 (car partial)
            else:
                amount = np.random.lognormal(mean=np.log(3000), sigma=0.3)  # 2000-5000 (full tank)
                
        elif category == 'Entertainment':
            # Entertainment: 100-1500, concentrated around movie/event tickets
            rand = np.random.random()
            if rand < 0.50:
                amount = np.random.lognormal(mean=np.log(200), sigma=0.4)  # 100-400 (tickets)
            elif rand < 0.80:
                amount = np.random.lognormal(mean=np.log(600), sigma=0.5)  # 400-1000 (events)
            else:
                amount = np.random.lognormal(mean=np.log(300), sigma=0.6)  # 100-500 (gaming)
                
        elif category == 'Shopping':
            # Shopping: 200-8000, wide range for different items
            rand = np.random.random()
            if rand < 0.40:
                amount = np.random.lognormal(mean=np.log(500), sigma=0.7)  # 200-1000
            elif rand < 0.75:
                amount = np.random.lognormal(mean=np.log(1800), sigma=0.6)  # 1000-3000
            else:
                amount = np.random.lognormal(mean=np.log(5000), sigma=0.4)  # 3000-8000
                
        elif category == 'Healthcare':
            # Healthcare: 200-2000, consultation and medicine focused
            rand = np.random.random()
            if rand < 0.50:
                amount = np.random.lognormal(mean=np.log(400), sigma=0.5)  # 200-800 (consultations)
            elif rand < 0.80:
                amount = np.random.lognormal(mean=np.log(300), sigma=0.4)  # 100-500 (medicines)
            else:
                amount = np.random.lognormal(mean=np.log(1000), sigma=0.5)  # 500-2000 (tests)
                
        elif category == 'Education':
            # Education: 500-15000, varied based on course type
            rand = np.random.random()
            if rand < 0.40:
                amount = np.random.lognormal(mean=np.log(1500), sigma=0.6)  # 500-3000 (online)
            elif rand < 0.75:
                amount = np.random.lognormal(mean=np.log(4000), sigma=0.5)  # 2000-8000 (tuition)
            else:
                amount = np.random.lognormal(mean=np.log(8000), sigma=0.4)  # 5000-15000 (institutional)
                
        elif category == 'Transport':
            # Transport: 30-800, based on ride type and distance
            rand = np.random.random()
            if rand < 0.50:
                amount = np.random.lognormal(mean=np.log(80), sigma=0.6)  # 30-200 (short rides)
            else:
                amount = np.random.lognormal(mean=np.log(400), sigma=0.6)  # 100-800 (app cabs)
                
        elif category == 'Utilities':
            # Utilities: 200-8000, concentrated around typical bill amounts
            rand = np.random.random()
            if rand < 0.25:
                amount = np.random.lognormal(mean=np.log(300), sigma=0.4)  # 100-500 (water)
            else:
                amount = np.random.lognormal(mean=np.log(2500), sigma=0.6)  # 800-5000 (electricity)
                
        else:  # Other category
            # Other: 100-2000, general miscellaneous payments
            amount = np.random.lognormal(mean=np.log(600), sigma=0.8)  # 100-2000

        # Apply age-based multiplier
        multiplier = age_multipliers.get(age_group, {}).get(category, 1.0)
        amount *= multiplier
        
        # Ensure minimum amount and round to nearest rupee
        amount = max(amount, 10)  # Minimum 10
        amount = min(amount, 100000)  # UPI limit 1 lakh
        amounts.append(round(amount))
    
    return np.array(amounts)

print("\n💰 Age-based amount generation function defined")
print("   10 merchant categories with realistic price ranges")
print("   5 age groups with spending multipliers")

In [0]:
def generate_upi_set(n_records=1000, start_date='2024-01-01', end_date='2024-12-31'):
    '''
    Generate synthetic UPI transaction dataset with 17 features
    
    Args:
        n_records: Number of transactions to generate
        start_date: Start date for transactions (YYYY-MM-DD)
        end_date: End date for transactions (YYYY-MM-DD)
    
    Returns:
        pandas DataFrame with 17 columns
    '''
    # Set seed for reproducibility
    np.random.seed(42)
    
    # Date range
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    date_range = (end - start).days
    
    # Transaction types
    transaction_type = np.random.choice(
        ['P2P', 'P2M', 'Bill Payment', 'Recharge'],
        size=n_records,
        p=[0.45, 0.35, 0.15, 0.05]
    )
    
    # Merchant categories
    merchant_categories = np.random.choice(
        ['Food', 'Grocery', 'Fuel', 'Entertainment', 'Shopping', 
         'Healthcare', 'Education', 'Transport', 'Utilities', 'Other'],
        size=n_records,
        p=[0.15, 0.20, 0.10, 0.08, 0.12, 0.05, 0.03, 0.08, 0.09, 0.10]
    )
    
    # Age groups
    sender_age = np.random.choice(
        ['18-25', '26-35', '36-45', '46-55', '56+'],
        size=n_records,
        p=[0.25, 0.35, 0.25, 0.10, 0.05]
    )
    
    # Build dataset
    data = {
        'txn_id': [f'TXN{str(i).zfill(10)}' for i in range(1, n_records + 1)],
        
        'timestamp': [start + timedelta(
            days=int(np.random.randint(0, date_range)),
            hours=int(np.random.choice(range(24), p=hourly_distribution())),
            minutes=int(np.random.randint(0, 60)),
            seconds=int(np.random.randint(0, 60))
        ) for _ in range(n_records)],
        
        'txn_type': transaction_type,
        'merchant_category': merchant_categories,
        'amount_inr': get_amt(merchant_categories, sender_age),
        
        # Transaction status
        'txn_status': np.random.choice(
            ['SUCCESS', 'FAILED'],
            size=n_records,
            p=[0.95, 0.05]
        ),
        
        # Demographics
        'sender_age_group': sender_age,
        'receiver_age_group': np.random.choice(
            ['18-25', '26-35', '36-45', '46-55', '56+'],
            size=n_records,
            p=[0.25, 0.35, 0.25, 0.10, 0.05]
        ),
        
        # Geographic distribution (matches Indian population)
        'sender_state': np.random.choice(
            ['Maharashtra', 'Karnataka', 'Delhi', 'Tamil Nadu', 'West Bengal', 
             'Gujarat', 'Rajasthan', 'Uttar Pradesh', 'Andhra Pradesh', 'Telangana'],
            size=n_records,
            p=[0.15, 0.12, 0.10, 0.10, 0.08, 0.08, 0.08, 0.12, 0.08, 0.09]
        ),
        
        # Banking (matches RBI market share)
        'sender_bank': np.random.choice(
            ['SBI', 'HDFC', 'ICICI', 'Axis', 'PNB', 'Kotak', 'IndusInd', 'Yes Bank'],
            size=n_records,
            p=[0.25, 0.15, 0.12, 0.10, 0.10, 0.08, 0.10, 0.10]
        ),
        'receiver_bank': np.random.choice(
            ['SBI', 'HDFC', 'ICICI', 'Axis', 'PNB', 'Kotak', 'IndusInd', 'Yes Bank'],
            size=n_records,
            p=[0.25, 0.15, 0.12, 0.10, 0.10, 0.08, 0.10, 0.10]
        ),
        
        # Device and network
        'device_type': np.random.choice(
            ['Android', 'iOS', 'Web'],
            size=n_records,
            p=[0.75, 0.20, 0.05]
        ),
        'network_type': np.random.choice(
            ['4G', '5G', 'WiFi', '3G'],
            size=n_records,
            p=[0.60, 0.25, 0.10, 0.05]
        ),
        
        # Fraud flag (0.2% fraud rate)
        'is_fraud': np.random.choice(
            [0, 1],
            size=n_records,
            p=[0.998, 0.002]
        )
    }
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Derive temporal features
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.day_name()
    df['is_weekend'] = (df['timestamp'].dt.weekday >= 5).astype(int)
    
    return df

print("\n🏗️ Main generation function defined")
print("   Generates 17 features per transaction")
print("   Ready to create dataset")

In [0]:
print("\n" + "="*70)
print("📊 GENERATING 500,000 UPI TRANSACTIONS")
print("="*70)

# Generate the dataset
df = generate_upi_set(
    n_records=500000,
    start_date='2024-01-01',
    end_date='2024-12-31'
)

print("\n✅ Dataset generated successfully!")
print("\n📈 Dataset Statistics:")
print(f"   Shape: {df.shape}")
print(f"   Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"   Total transaction value: ₹{df['amount_inr'].sum():,.0f}")
print(f"   Average transaction: ₹{df['amount_inr'].mean():,.0f}")
print(f"   Median transaction: ₹{df['amount_inr'].median():,.0f}")

print("\n🎯 Fraud Distribution:")
print(f"   Legitimate: {(df['is_fraud'] == 0).sum():,} ({(df['is_fraud'] == 0).sum() / len(df) * 100:.2f}%)")
print(f"   Fraud: {(df['is_fraud'] == 1).sum():,} ({(df['is_fraud'] == 1).sum() / len(df) * 100:.2f}%)")

print("\n🏦 Banking Distribution:")
print(df['sender_bank'].value_counts())

print("\n🌍 State Distribution:")
print(df['sender_state'].value_counts())

print("\n⏰ Peak Hours:")
top_hours = df['hour_of_day'].value_counts().head(5)
for hour, count in top_hours.items():
    print(f"   {hour}:00 - {count:,} transactions ({count/len(df)*100:.2f}%)")

# Display sample
print("\n📋 Sample Transactions:")
display(df.head(10))

In [0]:
print("\n" + "="*70)
print("💾 SAVING TO DELTA TABLE")
print("="*70)

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Save to Delta table (overwrite mode)
table_name = "workspace.bronze.fraud_data"
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✅ Data saved successfully to: {table_name}")

# Verify the save
verify_df = spark.table(table_name)
record_count = verify_df.count()
fraud_count = verify_df.filter("is_fraud = 1").count()

print("\n🔍 Verification:")
print(f"   Total records in table: {record_count:,}")
print(f"   Fraud records: {fraud_count:,}")
print(f"   Fraud rate: {fraud_count/record_count*100:.3f}%")

print("\n📊 Table Schema:")
verify_df.printSchema()

In [0]:
print("\n" + "="*70)
print("🔴 SAMPLE FRAUD TRANSACTIONS")
print("="*70)

# Get fraud transactions
fraud_samples = spark.sql("""
    SELECT 
        txn_id,
        timestamp,
        txn_type,
        merchant_category,
        amount_inr,
        txn_status,
        sender_age_group,
        sender_state,
        sender_bank,
        receiver_bank,
        device_type,
        hour_of_day,
        day_of_week
    FROM workspace.bronze.fraud_data
    WHERE is_fraud = 1
    ORDER BY amount_inr DESC
    LIMIT 20
""")

print("\n🎯 Top 20 Fraud Transactions by Amount:")
display(fraud_samples)

print("\n" + "="*70)
print("✅ DATA GENERATION COMPLETE!")
print("="*70)
print("\n📊 Dataset Summary:")
print(f"   Total Records: 500,000")
print(f"   Fraud Cases: ~1,000 (0.2%)")
print(f"   Time Period: Full Year 2024")
print(f"   Features: 17 total (14 raw + 3 derived)")
print("\n🎯 Next Steps:")
print("   1. Run baseline model training (01_train_model)")
print("   2. Expect higher accuracy with 2x more data")
print("   3. More fraud cases = better model learning")
print("\n" + "="*70)